# LKH-3 — Lin-Kernighan-Helsgott v3

## Description

LKH-3 est l'heuristique **Lin-Kernighan-Helsgott v3** (Helsgott, 2000+), état de l'art classique pour le TSP.

| Composant | Description |
|-----------|-------------|
| **Plus proche voisin** | Construction initiale gloutonne |
| **Listes de candidats** | k plus proches voisins par ville — restreint les échanges |
| **2-opt** | Supprime 2 arêtes et reconnecte mieux |
| **Or-opt** | Déplace des segments de 1, 2 ou 3 villes |
| **Double-bridge** | Perturbation 4-opt impossible à défaire par recherche séquentielle |
| **ILK** | Iterated Lin-Kernighan : perturbation → optimisation locale |

## Formules clés

**Gain 2-opt** entre les arêtes $(t_1,t_2)$ et $(t_3,t_4)$ :

$$G = d(t_1,t_2) + d(t_3,t_4) - d(t_1,t_3) - d(t_2,t_4)$$

Avec les candidats triés par distance croissante, on stoppe dès que $d(t_1,t_3) \geq d(t_1,t_2)$.

**Gain Or-opt** (segment $[s_0,\ldots,s_{l-1}]$ entre $a$–$b$, déplacé vers $u$–$v$) :

$$G = d(a,s_0) + d(s_{l-1},b) + d(u,v) - d(a,b) - d(u,s_0) - d(s_{l-1},v)$$

**Double-bridge** : $A + B + C + D \rightarrow A + C + B + D$


In [ ]:
import json
import numpy as np
import time
from pathlib import Path
from typing import List, Optional, Tuple

import time
import pandas as pd
import matplotlib.pyplot as plt

## Contraintes TSPTW-D intégrées

Les deux contraintes du **livrable_1_modelisation** sont maintenant intégrées dans `LKH3Solver` :

### Contrainte 1 — Fenêtres temporelles cycliques `[a_i, b_i]`

| Paramètre | Description |
|-----------|-------------|
| `time_windows` | array `(n+1, 2)` — `[(a_0, b_0), (a_1, b_1), …]` |
| `service_times` | array `(n+1,)` — `s_i` (durée de service en minutes) |
| `horizon` | durée d'un cycle (ex : 1440 min = 24h) |

**Logique d'attente** (`_next_service_time`) :

| Situation à l'arrivée | Comportement |
|---|---|
| `a_i ≤ τ_i ≤ b_i` | Départ immédiat : $D_i = \tau_i + s_i$ |
| `τ_i < a_i` (arrivée avant ouverture) | Attente jusqu'à l'ouverture : $D_i = a_i + s_i$ |
| `τ_i > b_i` (arrivée après fermeture) | **Attente jusqu'à la réouverture du prochain cycle** : $D_i = a_i + k \cdot H + s_i$ avec $k = \lceil (\tau_i - b_i) / H \rceil$ |

> Exemple : arrivée à 22h, fenêtre [8h, 20h], horizon H = 1440 min → k=1, prochain départ = 8h + 1440 min = 8h le lendemain.

Cette attente est coûteuse (elle augmente τ_retour) mais ne rend **jamais** la tournée infaisable. L'optimiseur est ainsi incité à éviter naturellement les arrivées hors fenêtre.

**Propagation temporelle** (`_propagate`) :
$$D_k = \texttt{next\_service\_time}(\tau_k,\ \sigma_k)$$
$$\tau_{k+1} = D_k + c_{\sigma_k, \sigma_{k+1}}(D_k)$$

### Contrainte 2 — Coûts dynamiques (perturbations)

| Paramètre | Description |
|-----------|-------------|
| `perturbations` | liste de `{arc, t_start, t_end, alpha}` |

**Coût dynamique** (`_c`) :
$$c_{ij}(t) = c_{ij}^{\text{base}} \times \alpha \quad \text{si } t \in [t_{\text{début}}, t_{\text{fin}}], \text{ sinon } c_{ij}^{\text{base}}$$

### Impact sur l'algorithme

- `_nn_tour` : choisit le voisin qui **minimise le coût effectif** (distance + attente éventuelle)
- `_two_opt_improve` / `_or_opt_improve` : comparent les durées totales τ_retour incluant les attentes de cycle
- `solve` : minimise $\tau_{\text{retour}}$, les attentes hors-fenêtre sont pénalisées implicitement par leur coût en temps


In [ ]:
class LKH3Solver:
    """
    Iterated Lin-Kernighan heuristic (LKH-3 style) — TSPTW-D variant.

    Contraintes intégrées (livrable_1_modelisation) :
    -------------------------------------------------
    1. Fenêtres temporelles cycliques [a_i, b_i] avec temps de service s_i.
       - Le véhicule peut arriver en avance et attendre l'ouverture.
       - Si le véhicule arrive APRÈS b_i (fenêtre fermée), il attend la
         réouverture au prochain cycle (t + horizon) : le coût est réel
         mais la tournée reste faisable. Cette attente est comptabilisée
         dans la durée totale τ_retour que l'on minimise.
         Exemple : arrivée à 22h, fenêtre [8h, 20h], horizon=1440 min →
                   on attend jusqu'à 8h le lendemain (attente = 10h).
    2. Coûts dynamiques : c_ij(t) = c_base_ij * alpha   si t ∈ [t_début, t_fin]
                                   = c_base_ij           sinon.
       La perturbation s'applique dans les deux sens (symétrique).

    Pipeline
    --------
    1. Nearest-Neighbour construction  (aware des fenêtres)
    2. 2-opt  (candidate-list restricted, coûts dynamiques)
    3. Or-opt (segments 1, 2, 3 villes, coûts dynamiques)
    4. Double-bridge perturbation  ->  goto 2
    """

    def __init__(
        self,
        coords,
        time_windows=None,   # list/array (n+1, 2)  : [(a0,b0), (a1,b1), ...]
        service_times=None,  # list/array (n+1,)    : [s0, s1, ...]
        perturbations=None,  # list of dicts {arc, t_start, t_end, alpha}
        scale: float = 1.0,  # multiplicateur coords→minutes (ex: 200)
        horizon: float = float("inf"),  # période de répétition des fenêtres (ex: 1440 min)
        k: int = 10,
        seed: int = 42,
    ):
        self.coords = np.asarray(coords, dtype=np.float64)
        self.n      = len(self.coords)
        self.k      = min(k, max(1, self.n - 1))
        self.rng    = np.random.default_rng(seed)
        self.scale   = scale
        self.horizon = horizon  # durée d'un cycle (inf = fenêtre unique stricte)

        # Fenêtres temporelles : par défaut [0, +inf]
        INF = float("inf")
        if time_windows is None:
            self.tw = np.array([[0.0, INF]] * self.n)
        else:
            self.tw = np.asarray(time_windows, dtype=np.float64)

        # Temps de service : par défaut 0
        if service_times is None:
            self.st = np.zeros(self.n)
        else:
            self.st = np.asarray(service_times, dtype=np.float64)

        # Perturbations
        self.perturbations = perturbations or []

        # Matrice de distances de base (coords normalisées × scale)
        self.dist_base  = self._compute_distances()
        self.candidates = self._build_candidates()

        self.converged:     bool      = False
        self.early_stopped: bool      = False
        self.best_tour:     List[int] = []
        self.best_length:   float     = float("inf")
        self.elapsed:       float     = 0.0

    # ------------------------------------------------------------------
    # Setup
    # ------------------------------------------------------------------

    def _compute_distances(self) -> np.ndarray:
        diff = self.coords[:, None, :] - self.coords[None, :, :]
        return np.sqrt((diff ** 2).sum(axis=-1)) * self.scale

    def _build_candidates(self) -> np.ndarray:
        """k nearest neighbours for each city (self excluded), sorted by distance."""
        return np.argsort(self.dist_base, axis=1)[:, 1 : self.k + 1]

    # ------------------------------------------------------------------
    # Contrainte 2 — Coût dynamique c_ij(t)
    # ------------------------------------------------------------------

    def _c(self, i: int, j: int, t: float) -> float:
        """
        Coût de transit de i vers j en partant à l'instant t.
        Applique le facteur alpha des perturbations actives à l'instant t.
        Les perturbations sont symétriques : arc (i,j) == arc (j,i).
        """
        base = self.dist_base[i, j]
        for p in self.perturbations:
            ai, aj = p["arc"]
            if (ai == i and aj == j) or (ai == j and aj == i):
                if p["t_start"] <= t <= p["t_end"]:
                    return base * p["alpha"]
        return base

    # ------------------------------------------------------------------
    # Contrainte 1 — Propagation temporelle avec fenêtres cycliques
    # ------------------------------------------------------------------

    
    def _next_service_time(self, t_arrive: float, city: int) -> float:
        """
        Calcule l'instant de départ D_i d'une ville, en tenant compte
        des fenêtres temporelles cycliques.
        """
        a_i = self.tw[city, 0]
        b_i = self.tw[city, 1]
        H = self.horizon

        # 1. Cas simple : Arrivée pendant ou avant la fenêtre du cycle 0.
        #    Cette condition gère aussi le cas où b_i = inf (fenêtre toujours ouverte).
        if t_arrive <= b_i + 1e-9:
            return max(t_arrive, a_i) + self.st[city]

        # 2. Arrivée après la fermeture du cycle 0 (t_arrive > b_i).
        #    Si l'horizon est infini, il n'y a pas d'autre cycle à attendre.
        if H == float("inf"):
            # On repart dès que possible (fenêtre souple)
            return t_arrive + self.st[city]

        # 3. Calcul du cycle k >= 1 pour les fenêtres répétitives.
        #    On cherche k tel que t_arrive <= b_i + k*H.
        import math
        # On sait ici que t_arrive > b_i et b_i est fini, donc k sera un nombre fini >= 1.
        k = math.ceil((t_arrive - b_i) / H)
        k = max(1, k)

        # Début du service : soit à l'arrivée (si on tombe dans la fenêtre du cycle k),
        # soit à l'ouverture du cycle k (si on est en avance sur le cycle k).
        open_time = a_i + k * H
        start_service = max(t_arrive, open_time)

        return start_service + self.st[city]

    def _propagate(self, tour: List[int]) -> Tuple[bool, float, List[float]]:
        """
        Propage les temps le long de la tournée avec fenêtres cycliques.

        Relation de récurrence étendue :
            D_k     = _next_service_time(τ_k, σ_k)
            τ_{k+1} = D_k + c(σ_k, σ_{k+1}, D_k)

        Une arrivée après b_i n'est PLUS infaisable : le véhicule attend
        la réouverture (prochain cycle), ce qui alourdit τ_retour et
        pousse naturellement l'optimiseur à éviter ces attentes.

        Retourne
        --------
        (True, total_duration, arrival_times)
            total_duration : τ_retour (toujours fini, jamais +inf)
            arrival_times  : liste des τ_k (len = n+2, le dernier = retour dépôt)
            late_flags     : stocké dans arrivals comme tuple (τ, hors_fenêtre)
                             → ici on retourne simplement les τ pour compatibilité
        """
        route    = tour + [tour[0]]
        t        = 0.0
        arrivals = [0.0]

        for k in range(len(route) - 1):
            city      = route[k]
            depart    = self._next_service_time(t, city)
            next_city = route[k + 1]
            travel    = self._c(city, next_city, depart)
            t         = depart + travel
            arrivals.append(t)

        return True, t, arrivals

    def _tour_length(self, tour: List[int]) -> float:
        if tour[0] != 0:
            idx = tour.index(0)
            tour = tour[idx:] + tour[:idx]
        """
        Durée totale de la tournée (τ_retour au dépôt), fenêtres cycliques.
        N'est jamais +inf : une attente hors-fenêtre alourdit simplement le coût.
        """
        _, duration, _ = self._propagate(tour)
        return duration

    # ------------------------------------------------------------------
    # Tour utilities
    # ------------------------------------------------------------------

    def _nn_tour(self, start: Optional[int] = None) -> List[int]:
        """
        Nearest-neighbour construction avec fenêtres temporelles cycliques.

        Stratégie de sélection : minimise le coût effectif d'atteindre
        chaque voisin non visité, en incluant l'éventuelle attente due
        à une fenêtre fermée (attente jusqu'au prochain cycle).
        Parmi les voisins candidats (liste de k plus proches), on choisit
        celui qui minimise  D_voisin = _next_service_time(τ_arrivée, voisin).
        """
        n = self.n
        if start is None:
            start = 0

        visited = np.zeros(n, dtype=bool)
        tour    = [start]
        visited[start] = True
        t = 0.0

        for _ in range(n - 1):
            city   = tour[-1]
            depart = self._next_service_time(t, city)

            # Évalue tous les voisins non visités
            best_cost = float("inf")
            best_nxt  = -1
            for nxt in range(n):
                if visited[nxt]:
                    continue
                travel   = self._c(city, nxt, depart)
                arr_nxt  = depart + travel
                # Coût effectif = arrivée + attente éventuelle jusqu'au départ
                eff_cost = self._next_service_time(arr_nxt, nxt)
                if eff_cost < best_cost:
                    best_cost = eff_cost
                    best_nxt  = nxt

            travel = self._c(city, best_nxt, depart)
            t      = depart + travel
            tour.append(best_nxt)
            visited[best_nxt] = True

        return tour

    # ------------------------------------------------------------------
    # Évaluation incrémentale d'un 2-opt sous TSPTW-D
    # ------------------------------------------------------------------

    def _eval_two_opt(self, tour: List[int], i: int, j: int) -> float:
        """
        Évalue la durée de la tournée après échange 2-opt (i, j).
        Retourne +inf si la tournée résultante est infaisable.
        """
        n   = len(tour)
        new = tour[:i+1] + tour[i+1:j+1][::-1] + tour[j+1:]
        return self._tour_length(new)

    # ------------------------------------------------------------------
    # 2-opt (candidate-list restricted, TSPTW-D aware)
    # ------------------------------------------------------------------

    def _two_opt_improve(self, tour: List[int]) -> List[int]:
        """
        2-opt avec coûts dynamiques et validation des fenêtres temporelles.

        La borne d'élagage utilise dist_base (distances statiques) pour
        l'early-exit, mais l'évaluation du gain réel passe par _tour_length
        (qui intègre les perturbations et les fenêtres).
        """
        n    = self.n
        t    = list(tour)
        best_len = self._tour_length(t)

        city_pos = np.empty(n, dtype=np.int32)
        for idx, c in enumerate(t):
            city_pos[c] = idx

        improved = True
        while improved:
            improved = False
            for i in range(n):
                t1  = t[i]
                t2  = t[(i + 1) % n]
                d12 = self.dist_base[t1, t2]

                for t3 in self.candidates[t1]:
                    # Early-exit basé sur distance statique (heuristique)
                    if self.dist_base[t1, t3] >= d12:
                        break
                    j  = int(city_pos[t3])
                    t4 = t[(j + 1) % n]
                    if t3 == t2 or t4 == t1:
                        continue

                    # Tentative de move 2-opt
                    if i < j:
                        new_t = t[:i+1] + t[i+1:j+1][::-1] + t[j+1:]
                    else:
                        new_t = t[:j+1] + t[j+1:i+1][::-1] + t[i+1:]

                    new_len = self._tour_length(new_t)
                    if new_len < best_len - 1e-10:
                        t        = new_t
                        best_len = new_len
                        for idx, c in enumerate(t):
                            city_pos[c] = idx
                        improved = True
                        break
        return t

    # ------------------------------------------------------------------
    # Or-opt (segments 1, 2, 3 — TSPTW-D aware)
    # ------------------------------------------------------------------

    def _or_opt_improve(self, tour: List[int]) -> List[int]:
        """
        Or-opt avec coûts dynamiques et validation des fenêtres temporelles.
        Évalue chaque relocation via _tour_length (TSPTW-D complet).
        """
        n = self.n
        t = list(tour)

        for seg_len in (1, 2, 3):
            if n <= seg_len + 1:
                continue

            city_pos = {c: idx for idx, c in enumerate(t)}
            improved = True

            while improved:
                improved = False
                best_len = self._tour_length(t)

                for i in range(n):
                    seg     = [t[(i + s) % n] for s in range(seg_len)]
                    a       = t[(i - 1) % n]
                    b       = t[(i + seg_len) % n]
                    seg_set = set(seg)
                    if a == b:
                        continue

                    best_gain = 1e-10
                    best_u    = None
                    best_rev  = False

                    for u in self.candidates[seg[0]]:
                        if u in seg_set or u == a:
                            continue
                        v = t[(city_pos[u] + 1) % n]
                        if v in seg_set:
                            continue

                        # Forward
                        remaining = [c for c in t if c not in seg_set]
                        ins = remaining.index(u)
                        new_t = remaining[:ins+1] + seg + remaining[ins+1:]
                        new_len = self._tour_length(new_t)
                        g = best_len - new_len
                        if g > best_gain:
                            best_gain, best_u, best_rev = g, u, False

                        # Reversed
                        new_t_r = remaining[:ins+1] + seg[::-1] + remaining[ins+1:]
                        new_len_r = self._tour_length(new_t_r)
                        g_r = best_len - new_len_r
                        if g_r > best_gain:
                            best_gain, best_u, best_rev = g_r, u, True

                    if best_u is not None:
                        seg_out   = seg[::-1] if best_rev else seg
                        remaining = [c for c in t if c not in seg_set]
                        ins = remaining.index(best_u)
                        t   = remaining[:ins+1] + seg_out + remaining[ins+1:]
                        city_pos = {c: idx for idx, c in enumerate(t)}
                        improved = True
                        break
        return t

    # ------------------------------------------------------------------
    # Double-bridge perturbation (inchangée)
    # ------------------------------------------------------------------

    def _double_bridge(self, tour: List[int]) -> List[int]:
        """
        4-opt double-bridge : A + C + B + D au lieu de A + B + C + D.
        Ne peut pas être défait par une recherche séquentielle 2-opt ou 3-opt.
        """
        n = self.n
        if n < 8:
            return tour[:]
        a, b, c, d = sorted(self.rng.choice(n, size=4, replace=False).tolist())
        return (tour[:a+1]
                + tour[c+1:d+1]
                + tour[b+1:c+1]
                + tour[a+1:b+1]
                + tour[d+1:])

    # ------------------------------------------------------------------
    # Main solve — Iterated Lin-Kernighan (TSPTW-D)
    # ------------------------------------------------------------------

    def solve(
        self,
        n_restarts: int = 20,
        time_limit: Optional[float] = None,
    ) -> Tuple[List[int], float]:
        """
        Boucle ILK avec gestion des contraintes TSPTW-D.

        La durée minimisée est τ_retour (temps de retour au dépôt),
        conforme à la fonction objectif du livrable_1_modelisation §2.4.
        Les tournées infaisables (fenêtre violée) ont une durée = +inf
        et sont automatiquement rejetées.

        Retourne
        --------
        (best_tour, best_length)   — best_length = τ_retour en minutes
        """
        t0 = time.perf_counter()
        self.converged     = False
        self.early_stopped = False

        if self.n == 1:
            self.best_tour, self.best_length = [0], 0.0
            self.elapsed  = 0.0
            self.converged = True
            return self.best_tour, self.best_length

        best     = self._nn_tour(start=0)
        best     = self._two_opt_improve(best)
        best     = self._or_opt_improve(best)
        best_len = self._tour_length(best)
        no_impr  = 0

        for _ in range(n_restarts):
            if time_limit and (time.perf_counter() - t0) > time_limit:
                self.early_stopped = True
                break
            t      = self._double_bridge(best)
            t      = self._two_opt_improve(t)
            t      = self._or_opt_improve(t)
            length = self._tour_length(t)
            if length < best_len - 1e-10:
                best, best_len, no_impr = t, length, 0
            else:
                no_impr += 1

        if no_impr == n_restarts:
            self.converged = True

        self.best_tour   = best
        self.best_length = best_len
        self.elapsed     = time.perf_counter() - t0
        # Assurer que la tournée commence par le dépôt (0)
        idx = best.index(0)
        self.best_tour   = best[idx:] + best[:idx]
        self.best_length = best_len
        self.elapsed     = time.perf_counter() - t0
        return self.best_tour, self.best_length


## Oracle exact — Held-Karp (n ≤ 12)


In [ ]:
def held_karp(dist: np.ndarray) -> Tuple[List[int], float]:
    """
    Exact TSP via Held-Karp DP.  O(n^2 * 2^n).
    Only feasible for n <= 15.
    """
    n    = len(dist)
    assert n <= 15, f"n={n} is too large for Held-Karp"
    INF  = float("inf")
    size = 1 << n
    dp     = [[INF] * n for _ in range(size)]
    parent = [[-1]  * n for _ in range(size)]
    dp[1][0] = 0.0

    for mask in range(1, size):
        if not (mask & 1):
            continue
        for u in range(n):
            if not (mask & (1 << u)) or dp[mask][u] == INF:
                continue
            for v in range(n):
                if mask & (1 << v):
                    continue
                nm = mask | (1 << v)
                nc = dp[mask][u] + dist[u, v]
                if nc < dp[nm][v]:
                    dp[nm][v] = nc
                    parent[nm][v] = u

    full = size - 1
    best_cost, last = INF, -1
    for u in range(1, n):
        c = dp[full][u] + dist[u, 0]
        if c < best_cost:
            best_cost, last = c, u

    tour, mask, u = [], full, last
    while u != -1:
        tour.append(u)
        prev = parent[mask][u]
        mask ^= (1 << u)
        u = prev
    tour.reverse()
    return tour, best_cost


## Benchmark


In [ ]:
EXACT_THRESHOLD = 12   # use Held-Karp exact optimal for n <= this

def _time_limit(n: int) -> Optional[float]:
    if n <= 100:    return None
    if n <= 1_000:  return 30.0
    if n <= 10_000: return 60.0
    return 120.0

def _n_restarts(n: int) -> int:
    if n <= 100:    return 50
    if n <= 1_000:  return 20
    if n <= 10_000: return 5
    return 2


def benchmark_size(n: int, n_runs: int = 20, seed: int = 0) -> List[dict]:
    """Run LKH3 on n_runs random Euclidean TSP instances of size n."""
    rng     = np.random.default_rng(seed)
    results = []

    for run in range(n_runs):
        if n == 1:
            results.append({
                "n": 1, "run": run, "length": 0.0, "optimal": 0.0,
                "gap": 0.0, "elapsed": 0.0,
                "converged": True, "early_stopped": False,
            })
            continue

        coords = rng.random((n, 2))
        solver = LKH3Solver(coords, k=min(10, n - 1), seed=seed + run)
        solver.solve(n_restarts=_n_restarts(n), time_limit=_time_limit(n))

        optimal = None
        if n <= EXACT_THRESHOLD:
            _, optimal = held_karp(solver.dist_base)

        results.append({
            "n":             n,
            "run":           run,
            "length":        solver.best_length,
            "optimal":       optimal,
            "elapsed":       solver.elapsed,
            "converged":     solver.converged,
            "early_stopped": solver.early_stopped,
        })

    # For n > EXACT_THRESHOLD: best-of-runs is the reference
    if n > EXACT_THRESHOLD:
        ref = min(r["length"] for r in results)
        for r in results:
            r["optimal"] = ref

    for r in results:
        opt = r["optimal"]
        r["gap"]             = (r["length"] - opt) / opt * 100 if opt > 1e-10 else 0.0
        r["success"]         = r["gap"] <= 1.0
        # converged=True but gap > 1%  -> algo missed the optimum
        r["non_detection"]   = r["converged"] and (r["gap"] > 1.0)
        # early_stopped=True but gap <= 1%  -> stopped prematurely despite good solution
        r["false_detection"] = r["early_stopped"] and (r["gap"] <= 1.0)

    return results


def summarise(results_by_n: dict) -> pd.DataFrame:
    rows = []
    for n, res in sorted(results_by_n.items()):
        gaps = [r["gap"]     for r in res]
        t_ms = [r["elapsed"] * 1_000 for r in res]
        rows.append({
            "n":                  n,
            "% reussite":         f"{np.mean([r['success']         for r in res]) * 100:.0f} %",
            "% non-detection":    f"{np.mean([r['non_detection']   for r in res]) * 100:.0f} %",
            "% fausse detection": f"{np.mean([r['false_detection'] for r in res]) * 100:.0f} %",
            "Gap moyen (%)":      f"{np.mean(gaps):.3f}",
            "Gap max (%)":        f"{np.max(gaps):.3f}",
            "Temps moyen (ms)":   f"{np.mean(t_ms):.1f} +/- {np.std(t_ms):.1f}",
        })
    return pd.DataFrame(rows).set_index("n")


### Graphes aléatoires (1 → 1 000 noeuds)

> **Note :** n = 100 000 dépasse les capacités du Python pur (LKH-3 natif est en C).
> Le benchmark s'arrête à **n = 1 000** avec un budget temps réduit.


In [ ]:
SIZES  = [1, 10]
N_RUNS = 20

results_by_n = {}
for n in SIZES:
    print(f"Benchmarking n={n:>6} ...", end=" ", flush=True)
    t0 = time.perf_counter()
    results_by_n[n] = benchmark_size(n, n_runs=N_RUNS, seed=0)
    print(f"done ({time.perf_counter() - t0:.1f}s)")

df_random = summarise(results_by_n)
print("\n=== Benchmark — graphes aleatoires ===")
df_random


### Dataset `datasets/tsptwd_n*.json`

Chaque instance JSON expose un dépôt + $n$ clients avec coordonnées $(x, y) \in [0,1]^2$.  
Les coûts sont rapportés avec le **facteur ×200** (`scale = meta["scale"]`) pour être directement comparables à `tour_cost` de POMO.

In [ ]:
DATASETS_DIR = Path("datasets")


def load_json_instance(path: Path):
    """
    Charge un fichier tsptwd_n*.json (format TSPTW-D).

    Retourne
    --------
    coords        : ndarray (n+1, 2)  — dépôt en index 0, puis clients
    time_windows  : ndarray (n+1, 2)  — [(a_i, b_i)] pour chaque nœud
    service_times : ndarray (n+1,)    — s_i pour chaque nœud
    perturbations : list of dict      — [{arc, t_start, t_end, alpha}, ...]
    meta          : dict              — n_clients, scale, horizon, seed
    """
    with open(path) as f:
        data = json.load(f)

    nodes = [data["depot"]] + data["clients"]
    coords = np.array([[nd["x"], nd["y"]] for nd in nodes], dtype=np.float64)

    INF = float("inf")
    time_windows  = np.array(
        [[nd["a"] if nd["a"] is not None else 0.0,
          nd["b"] if nd["b"] is not None else INF]
         for nd in nodes],
        dtype=np.float64,
    )
    service_times = np.array([nd["service"] for nd in nodes], dtype=np.float64)
    perturbations = data.get("perturbations", [])
    
    return coords, time_windows, service_times, perturbations, data["meta"]


def benchmark_datasets(
    datasets_dir: Path,
    skip_n: int = 200,
) -> pd.DataFrame:
    """
    Exécute LKH3-TSPTWD sur toutes les instances JSON du dossier datasets/.

    Les durées sont exprimées directement en minutes (les coords sont
    multipliées par scale=200 à l'initialisation du solver).
    Les instances avec n >= skip_n sont ignorées (trop lentes en Python pur).
    """
    files = sorted(
        datasets_dir.glob("tsptwd_n*.json"),
        key=lambda p: int(p.stem.split("_n")[1]),
    )
    rows = []

    for path in files:
        coords, time_windows, service_times, perturbations, meta = load_json_instance(path)
        n     = meta["n_clients"]
        scale = meta["scale"]          # 200.0

        if n >= skip_n:
            print(f"  n={n:>6}  [SKIP] n >= {skip_n}")
            continue

        solver = LKH3Solver(
            coords,
            time_windows=time_windows,
            service_times=service_times,
            perturbations=perturbations,
            scale=scale,
            horizon=meta.get("horizon", float("inf")),
            k=min(10, len(coords) - 1),
            seed=42,
        )
        solver.solve(n_restarts=_n_restarts(n), time_limit=_time_limit(n))

        cost_lkh3 = round(solver.best_length, 1)   # déjà en minutes

        # Baseline NN depuis le dépôt (index 0)
        nn_tour  = solver._nn_tour(start=0)
        cost_nn  = round(solver._tour_length(nn_tour), 1)

        feasible = cost_lkh3 < float("inf")
        if cost_nn < float("inf") and feasible:
            gap_vs_nn = round((cost_lkh3 - cost_nn) / cost_nn * 100, 2)
        else:
            gap_vs_nn = float("nan")

        print(
            f"  n={n:>6}  LKH3={cost_lkh3:>9.1f} min"
            f"  NN={cost_nn:>9.1f} min"
            f"  gain={-gap_vs_nn if feasible else float('nan'):+.1f}%"
            f"  t={solver.elapsed:.2f}s"
            f"  {'OK' if feasible else 'INFEASIBLE'}"
        )

        rows.append({
            "n":              n,
            "cost_lkh3_min":  cost_lkh3,
            "cost_nn_min":    cost_nn,
            "feasible":       feasible,
            "gain_vs_nn_%":   round(-gap_vs_nn, 2) if feasible else float("nan"),
            "elapsed_s":      round(solver.elapsed, 3),
            "converged":      solver.converged,
            "early_stopped":  solver.early_stopped,
        })

    return pd.DataFrame(rows).set_index("n")


df_datasets = benchmark_datasets(DATASETS_DIR)
df_datasets


## Visualisation


In [ ]:
def plot_tour_tsptwd(
    coords: np.ndarray,
    tour: List[int],
    time_windows,
    arrival_times: List[float],
    perturbations=None,
    title: str = "",
    scale: float = 1.0,
) -> None:
    """
    Visualise une tournée TSPTW-D avec :
    - Arêtes perturbées en rouge
    - Fenêtres temporelles affichées sous chaque nœud
    - Heure d'arrivée réelle τ_i affichée
    """
    perturbations = perturbations or []
    n   = len(tour)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Graphe de la tournée ──────────────────────────────────────────────────
    ax = axes[0]
    route = tour + [tour[0]]
    xs = [coords[c, 0] for c in route]
    ys = [coords[c, 1] for c in route]

    # Arêtes : couleur selon perturbation
    for k in range(len(route) - 1):
        i, j = route[k], route[k+1]
        perturbed = any(
            (p["arc"][0] == i and p["arc"][1] == j) or
            (p["arc"][0] == j and p["arc"][1] == i)
            for p in perturbations
        )
        color = "crimson" if perturbed else "steelblue"
        lw    = 2.5    if perturbed else 1.2
        ax.annotate(
            "",
            xy=(coords[j, 0], coords[j, 1]),
            xytext=(coords[i, 0], coords[i, 1]),
            arrowprops=dict(arrowstyle="->", color=color, lw=lw),
        )

    ax.scatter(coords[tour, 0], coords[tour, 1], s=60, color="steelblue", zorder=3)
    ax.scatter(coords[tour[0], 0], coords[tour[0], 1], s=120, color="orange", zorder=4, label="Dépôt")

    # Labels : id, fenêtre, arrivée
    for rank, city in enumerate(tour):
        tw = time_windows[city]
        arr = arrival_times[rank] if rank < len(arrival_times) else float("nan")
        ax.annotate(
            f"v{city}\n[{tw[0]:.0f},{tw[1]:.0f}]\nτ={arr:.1f}",
            (coords[city, 0], coords[city, 1]),
            textcoords="offset points", xytext=(6, 6),
            fontsize=6.5,
        )

    ax.set_title(title or "Tournée LKH-3 TSPTW-D")
    ax.legend()
    ax.set_aspect("equal")

    # ── Diagramme de Gantt des arrivées ──────────────────────────────────────
    ax2 = axes[1]
    cmap = plt.cm.tab10
    for rank, city in enumerate(tour):
        tw  = time_windows[city]
        arr = arrival_times[rank] if rank < len(arrival_times) else float("nan")
        col = cmap(rank % 10)

        # Fenêtre temporelle
        ax2.barh(rank, tw[1] - tw[0], left=tw[0], height=0.5,
                 color=col, alpha=0.25, edgecolor="grey", linewidth=0.5)
        # Arrivée réelle
        if not np.isnan(arr):
            late = arr > tw[1]
            ax2.plot(arr, rank, marker="D", color="crimson" if late else col,
                     markersize=7, zorder=3,
                     label="Hors fenêtre (attente cycle)" if (late and rank == 0) else None)
            if late:
                # Visualiser l'attente jusqu'à la prochaine ouverture
                H = meta_d.get("horizon", float("inf")) if "meta_d" in dir() else float("inf")
                if H < float("inf"):
                    import math
                    k_cycle = math.ceil((arr - tw[1]) / H)
                    next_open = tw[0] + k_cycle * H
                    ax2.barh(rank, next_open - arr, left=arr, height=0.3,
                             color="crimson", alpha=0.35, hatch="//", label="Attente réouverture" if rank == 0 else None)
            ax2.axvline(arr, color=col, alpha=0.3, lw=0.8)

        ax2.text(-2, rank, f"v{city}", ha="right", va="center", fontsize=7)

    ax2.set_xlabel("Temps (minutes)")
    ax2.set_title("Fenêtres temporelles et heures d'arrivée")
    ax2.set_yticks([])
    ax2.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()


# ── Demo sur une instance JSON ────────────────────────────────────────────────

demo_path = next(Path("datasets").glob("tsptwd_n10.json"), None)
if demo_path is None:
    demo_path = next(Path("datasets").glob("tsptwd_n*.json"), None)

if demo_path:
    coords_d, tw_d, st_d, pert_d, meta_d = load_json_instance(demo_path)
    scale_d = meta_d["scale"]

    t0 = time.perf_counter()
    
    solver_d = LKH3Solver(
        coords_d,
        time_windows=tw_d,
        service_times=st_d,
        perturbations=pert_d,
        scale=scale_d,
        horizon=meta_d.get("horizon", float("inf")),
        k=min(10, len(coords_d) - 1),
        seed=42,
    )
    tour_d, len_d = solver_d.solve(n_restarts=30)
    _, _, arr_d   = solver_d._propagate(tour_d)
    
    elapsed = time.perf_counter() - t0
    
    print(f"Temps      : {elapsed:.3f}s")

    print(f"Instance : {demo_path.name}")
    print(f"Durée optimale (τ_retour) : {len_d:.1f} min")
    print(f"Ordre de visite : {' → '.join(f'v{c}' for c in tour_d + [tour_d[0]])}")
    print(f"Perturbations   : {pert_d}")

    plot_tour_tsptwd(
        coords_d, tour_d, tw_d, arr_d,
        perturbations=pert_d,
        title=f"LKH-3 TSPTW-D — {demo_path.name} — durée = {len_d:.1f} min",
        scale=scale_d,
    )
else:
    # Démo sur 20 villes aléatoires sans contraintes (comportement identique à l'original)
    rng_demo   = np.random.default_rng(7)
    demo_coord = rng_demo.random((20, 2))
    demo_sol   = LKH3Solver(demo_coord, k=10, seed=42)
    demo_tour, demo_len = demo_sol.solve(n_restarts=50)
    nn_len = demo_sol._tour_length(demo_sol._nn_tour(start=0))
    print(f"NN initial : {nn_len:.4f}")
    print(f"LKH-3      : {demo_len:.4f}  ({(nn_len-demo_len)/nn_len*100:.1f}% d'amélioration)")

    coords_arr = np.array(demo_coord)
    tw_fake    = np.array([[0.0, float("inf")]] * len(demo_coord))
    _, _, arr_demo = demo_sol._propagate(demo_tour)
    plot_tour_tsptwd(
        coords_arr, demo_tour, tw_fake, arr_demo,
        title=f"LKH-3 — 20 villes aléatoires — longueur = {demo_len:.3f}",
    )


In [ ]:

demo_path = next(Path("datasets").glob("tsptwd_n50.json"), None)
if demo_path is None:
    demo_path = next(Path("datasets").glob("tsptwd_n*.json"), None)

if demo_path:
    coords_d, tw_d, st_d, pert_d, meta_d = load_json_instance(demo_path)
    scale_d = meta_d["scale"]

    t0 = time.perf_counter()
    
    solver_d = LKH3Solver(
        coords_d,
        time_windows=tw_d,
        service_times=st_d,
        perturbations=pert_d,
        scale=scale_d,
        horizon=meta_d.get("horizon", float("inf")),
        k=min(10, len(coords_d) - 1),
        seed=42,
    )
    tour_d, len_d = solver_d.solve(n_restarts=30)
    _, _, arr_d   = solver_d._propagate(tour_d)
    
    elapsed = time.perf_counter() - t0
    
    print(f"Temps      : {elapsed:.3f}s")

    print(f"Instance : {demo_path.name}")
    print(f"Durée optimale (τ_retour) : {len_d:.1f} min")
    print(f"Ordre de visite : {' → '.join(f'v{c}' for c in tour_d + [tour_d[0]])}")
    print(f"Perturbations   : {pert_d}")

    plot_tour_tsptwd(
        coords_d, tour_d, tw_d, arr_d,
        perturbations=pert_d,
        title=f"LKH-3 TSPTW-D — {demo_path.name} — durée = {len_d:.1f} min",
        scale=scale_d,
    )
else:
    # Démo sur 20 villes aléatoires sans contraintes (comportement identique à l'original)
    rng_demo   = np.random.default_rng(7)
    demo_coord = rng_demo.random((20, 2))
    demo_sol   = LKH3Solver(demo_coord, k=10, seed=42)
    demo_tour, demo_len = demo_sol.solve(n_restarts=50)
    nn_len = demo_sol._tour_length(demo_sol._nn_tour(start=0))
    print(f"NN initial : {nn_len:.4f}")
    print(f"LKH-3      : {demo_len:.4f}  ({(nn_len-demo_len)/nn_len*100:.1f}% d'amélioration)")

    coords_arr = np.array(demo_coord)
    tw_fake    = np.array([[0.0, float("inf")]] * len(demo_coord))
    _, _, arr_demo = demo_sol._propagate(demo_tour)
    plot_tour_tsptwd(
        coords_arr, demo_tour, tw_fake, arr_demo,
        title=f"LKH-3 — 20 villes aléatoires — longueur = {demo_len:.3f}",
    )

In [ ]:

demo_path = next(Path("datasets").glob("tsptwd_n100.json"), None)
if demo_path is None:
    demo_path = next(Path("datasets").glob("tsptwd_n*.json"), None)

if demo_path:
    coords_d, tw_d, st_d, pert_d, meta_d = load_json_instance(demo_path)
    scale_d = meta_d["scale"]

    t0 = time.perf_counter()
    
    solver_d = LKH3Solver(
        coords_d,
        time_windows=tw_d,
        service_times=st_d,
        perturbations=pert_d,
        scale=scale_d,
        horizon=meta_d.get("horizon", float("inf")),
        k=min(10, len(coords_d) - 1),
        seed=42,
    )
    tour_d, len_d = solver_d.solve(n_restarts=30)
    _, _, arr_d   = solver_d._propagate(tour_d)
    
    elapsed = time.perf_counter() - t0
    
    print(f"Temps      : {elapsed:.3f}s")

    print(f"Instance : {demo_path.name}")
    print(f"Durée optimale (τ_retour) : {len_d:.1f} min")
    print(f"Ordre de visite : {' → '.join(f'v{c}' for c in tour_d + [tour_d[0]])}")
    print(f"Perturbations   : {pert_d}")

    plot_tour_tsptwd(
        coords_d, tour_d, tw_d, arr_d,
        perturbations=pert_d,
        title=f"LKH-3 TSPTW-D — {demo_path.name} — durée = {len_d:.1f} min",
        scale=scale_d,
    )
else:
    # Démo sur 20 villes aléatoires sans contraintes (comportement identique à l'original)
    rng_demo   = np.random.default_rng(7)
    demo_coord = rng_demo.random((20, 2))
    demo_sol   = LKH3Solver(demo_coord, k=10, seed=42)
    demo_tour, demo_len = demo_sol.solve(n_restarts=50)
    nn_len = demo_sol._tour_length(demo_sol._nn_tour(start=0))
    print(f"NN initial : {nn_len:.4f}")
    print(f"LKH-3      : {demo_len:.4f}  ({(nn_len-demo_len)/nn_len*100:.1f}% d'amélioration)")

    coords_arr = np.array(demo_coord)
    tw_fake    = np.array([[0.0, float("inf")]] * len(demo_coord))
    _, _, arr_demo = demo_sol._propagate(demo_tour)
    plot_tour_tsptwd(
        coords_arr, demo_tour, tw_fake, arr_demo,
        title=f"LKH-3 — 20 villes aléatoires — longueur = {demo_len:.3f}",
    )

In [ ]:

demo_path = next(Path("datasets").glob("tsptwd_n200.json"), None)
if demo_path is None:
    demo_path = next(Path("datasets").glob("tsptwd_n*.json"), None)

if demo_path:
    coords_d, tw_d, st_d, pert_d, meta_d = load_json_instance(demo_path)
    scale_d = meta_d["scale"]

    t0 = time.perf_counter()
    
    solver_d = LKH3Solver(
        coords_d,
        time_windows=tw_d,
        service_times=st_d,
        perturbations=pert_d,
        scale=scale_d,
        horizon=meta_d.get("horizon", float("inf")),
        k=min(10, len(coords_d) - 1),
        seed=42,
    )
    tour_d, len_d = solver_d.solve(n_restarts=30)
    _, _, arr_d   = solver_d._propagate(tour_d)
    
    elapsed = time.perf_counter() - t0
    
    print(f"Temps      : {elapsed:.3f}s")

    print(f"Instance : {demo_path.name}")
    print(f"Durée optimale (τ_retour) : {len_d:.1f} min")
    print(f"Ordre de visite : {' → '.join(f'v{c}' for c in tour_d + [tour_d[0]])}")
    print(f"Perturbations   : {pert_d}")

    plot_tour_tsptwd(
        coords_d, tour_d, tw_d, arr_d,
        perturbations=pert_d,
        title=f"LKH-3 TSPTW-D — {demo_path.name} — durée = {len_d:.1f} min",
        scale=scale_d,
    )
else:
    # Démo sur 20 villes aléatoires sans contraintes (comportement identique à l'original)
    rng_demo   = np.random.default_rng(7)
    demo_coord = rng_demo.random((20, 2))
    demo_sol   = LKH3Solver(demo_coord, k=10, seed=42)
    demo_tour, demo_len = demo_sol.solve(n_restarts=50)
    nn_len = demo_sol._tour_length(demo_sol._nn_tour(start=0))
    print(f"NN initial : {nn_len:.4f}")
    print(f"LKH-3      : {demo_len:.4f}  ({(nn_len-demo_len)/nn_len*100:.1f}% d'amélioration)")

    coords_arr = np.array(demo_coord)
    tw_fake    = np.array([[0.0, float("inf")]] * len(demo_coord))
    _, _, arr_demo = demo_sol._propagate(demo_tour)
    plot_tour_tsptwd(
        coords_arr, demo_tour, tw_fake, arr_demo,
        title=f"LKH-3 — 20 villes aléatoires — longueur = {demo_len:.3f}",
    )

In [ ]:

demo_path = next(Path("datasets").glob("tsptwd_n500.json"), None)
if demo_path is None:
    demo_path = next(Path("datasets").glob("tsptwd_n*.json"), None)

if demo_path:
    coords_d, tw_d, st_d, pert_d, meta_d = load_json_instance(demo_path)
    scale_d = meta_d["scale"]

    t0 = time.perf_counter()
    
    solver_d = LKH3Solver(
        coords_d,
        time_windows=tw_d,
        service_times=st_d,
        perturbations=pert_d,
        scale=scale_d,
        horizon=meta_d.get("horizon", float("inf")),
        k=min(10, len(coords_d) - 1),
        seed=42,
    )
    tour_d, len_d = solver_d.solve(n_restarts=30)
    _, _, arr_d   = solver_d._propagate(tour_d)
    
    elapsed = time.perf_counter() - t0
    
    print(f"Temps      : {elapsed:.3f}s")

    print(f"Instance : {demo_path.name}")
    print(f"Durée optimale (τ_retour) : {len_d:.1f} min")
    print(f"Ordre de visite : {' → '.join(f'v{c}' for c in tour_d + [tour_d[0]])}")
    print(f"Perturbations   : {pert_d}")

    plot_tour_tsptwd(
        coords_d, tour_d, tw_d, arr_d,
        perturbations=pert_d,
        title=f"LKH-3 TSPTW-D — {demo_path.name} — durée = {len_d:.1f} min",
        scale=scale_d,
    )
else:
    # Démo sur 20 villes aléatoires sans contraintes (comportement identique à l'original)
    rng_demo   = np.random.default_rng(7)
    demo_coord = rng_demo.random((20, 2))
    demo_sol   = LKH3Solver(demo_coord, k=10, seed=42)
    demo_tour, demo_len = demo_sol.solve(n_restarts=50)
    nn_len = demo_sol._tour_length(demo_sol._nn_tour(start=0))
    print(f"NN initial : {nn_len:.4f}")
    print(f"LKH-3      : {demo_len:.4f}  ({(nn_len-demo_len)/nn_len*100:.1f}% d'amélioration)")

    coords_arr = np.array(demo_coord)
    tw_fake    = np.array([[0.0, float("inf")]] * len(demo_coord))
    _, _, arr_demo = demo_sol._propagate(demo_tour)
    plot_tour_tsptwd(
        coords_arr, demo_tour, tw_fake, arr_demo,
        title=f"LKH-3 — 20 villes aléatoires — longueur = {demo_len:.3f}",
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sizes  = [n for n in sorted(results_by_n) if n > 1]
labels = [str(n) for n in sizes]

mean_gaps  = [np.mean([r["gap"]     for r in results_by_n[n]]) for n in sizes]
success_rt = [np.mean([r["success"] for r in results_by_n[n]]) * 100 for n in sizes]
mean_times = [np.mean([r["elapsed"] for r in results_by_n[n]]) * 1_000 for n in sizes]

axes[0].bar(labels, mean_gaps, color="steelblue")
axes[0].axhline(1.0, color="crimson", ls="--", label="seuil 1 %")
axes[0].set_title("Gap moyen (%) vs reference")
axes[0].set_xlabel("n") ; axes[0].set_ylabel("%") ; axes[0].legend()

axes[1].bar(labels, success_rt, color="seagreen")
axes[1].set_ylim(0, 105)
axes[1].set_title("% reussite (gap <= 1 %)")
axes[1].set_xlabel("n") ; axes[1].set_ylabel("%")

axes[2].bar(labels, mean_times, color="darkorange")
axes[2].set_yscale("log")
axes[2].set_title("Temps moyen par instance (ms)")
axes[2].set_xlabel("n") ; axes[2].set_ylabel("ms")

plt.suptitle("LKH-3 - benchmark graphes aleatoires", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nTableau recapitulatif :")
df_random
